# Run-book 198 — does mpc-fast-extract make datacube creation faster?

**What this proves:** for the same geometries, dates and bands, how long rise-eo takes to create
datacubes (download whole granules, then build) against mpc-fast-extract (stream only the blocks
it needs into a Zarr, then cut that into rise-eo cubes). It also says how closely the two sets of
cubes agree, pixel by pixel. It is the P0 spike of issue #198: the numbers decide whether an
mpc-fast-extract build backend gets a spec. Two workloads:
- **fields**: the 900 EuroCrops fields of `AT_2018_TRAIN` (the training case: many tiny cubes);
- **cells**: `AT_ROI` cut into grid cells (the inference case: a few hundred large cubes).

**What it does not show:** model results, CDSE, the AML cluster (both sides run on one machine),
ROIs across UTM zones (all of Austria is UTM 33N).

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs** (all estimates):
- your AML compute instance: Linux x86-64 with AVX2 (any Dv5), in West Europe next to the imagery;
- network to Microsoft Planetary Computer (anonymous, no credentials);
- disk: about 70 GB for the archive plus about 10 GB for the Zarr and the cubes, on a local disk;
- time: 30–45 min of one-time setup, then about 1–1.5 h of running, mostly the download and the
  rise-eo builds.

<!-- Rules for writing one (spec 24 amendment A1):
  - call rise-eo's public functions directly, keyword arguments spelled out;
  - small helpers only (~15 lines), defined just above their first use, with a docstring;
  - no environment variables, no try/except around a whole step, no result-file plumbing;
  - every step reads what it needs from disk + Settings, never from an earlier step's variables;
  - a plan cell (counts + GB, no transfer) before any step that moves more than ~1 GB;
  - a deliberate kill runs in a child process (os._exit would kill this kernel). -->

## Before you start

**One-time setup on the compute instance.** Open a terminal on it (Azure ML studio → Compute →
your compute instance → Terminal) and run these. They install rise-eo at this branch, Rust, and
mpc-fast-extract into rise-eo's `.venv`. Everything goes under your home folder, which is on the
instance's own disk; `~/cloudfiles` is a network share, so keep things out of it.

```bash
# 1. rise-eo at this branch, in a Python 3.11 venv (the compute instance ships conda)
conda create -y -p "$HOME/py311" python=3.11
git clone https://github.com/nasaharvest/rise-eo.git "$HOME/rise-eo"
cd "$HOME/rise-eo" && git checkout spike/p0-mpc-fast-extract-parity
"$HOME/py311/bin/python" -m venv .venv
.venv/bin/pip install -e ".[dev,local,s3,notebooks,grid,azure,aml,mpc,titiler,serving]" ipykernel

# 2. Rust, then mpc-fast-extract at the commit this run-book was written against
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
. "$HOME/.cargo/env"
git clone https://github.com/rohansaw/mpc-fast-extract.git "$HOME/mpc-fast-extract"
git -C "$HOME/mpc-fast-extract" checkout da6bf12
.venv/bin/pip install -e "$HOME/mpc-fast-extract/python[raster]"   # compiles the engine: ~5-15 min

# 3. find a local disk with ~120 GB free for OUT in Settings (often /mnt); not ~/cloudfiles
df -h
```

If the mpc-fast-extract clone asks for a login, the repository is private: ask Rohan for access.
If `pip` reports a version conflict, paste its message back instead of forcing it.

**Kernel — which Python runs the cells.** In Azure ML studio, open the compute instance in
**VS Code (Desktop)**, open the folder `~/rise-eo`, open this notebook, then **Select Kernel** →
**Python Environments** → **`.venv`**. The **Setup** cell below checks this and stops with a
message if the kernel is wrong.

**Code — which `rise-eo` gets imported.** The Setup cell puts **this notebook's own** `src/` first
on Python's import path and checks it worked.

**Stop / resume.** Interrupt the kernel (■) to stop. After a kernel restart, run **Setup**,
**Settings** and **Helpers** again, then continue at any step: every step reads what it needs from
disk. A timed step that already finished is skipped (its time is in `OUT/timings/`). To time a step
again, delete its file there. The download resumes where it stopped, but then its time covers only
the rest: for a clean time, delete `OUT/archive` and its timing file, and run Step 4 again.

**When you are done.** Paste back what the **Summary** cell prints (plus any failing cell's
message). Run the Summary even if a check failed: it reads whatever is on disk. Then
**Restart & Clear All Outputs** and save: this file is tracked in git, and a test refuses saved
outputs.

In [ ]:
# Setup -- which Python, and which rise-eo. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be rise-eo's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not rise-eo's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the rise-eo checkout that owns the venv

# 2. Which rise-eo? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("rise-eo    :", rise_eo.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else. `mpc_config` is an input too:
it is the mpc-fast-extract config that mirrors rise-eo's build, with a comment on each choice.

In [ ]:
# Settings
import datetime
import math
import os

import pandas as pd

from rise_eo import collections as rise_collections
from rise_eo.api import archive_catalog_filepath

OUT = Path("/mnt/rise-p0")   # a LOCAL disk of the compute instance (`df -h`), never ~/cloudfiles
NEED_GB = 120                # free space wanted at OUT before the download (estimate, with headroom)
MPC_FAST_EXTRACT = Path.home() / "mpc-fast-extract"   # the checkout pip-installed into .venv
MPC_FAST_EXTRACT_COMMIT = "da6bf12"                    # the version this run-book was written against

SHAPES = CHECKOUT / "notebooks" / "shapefiles"
LABELS = SHAPES / "AT_2018_TRAIN.geojson"   # 900 EuroCrops fields: the training case
ROI = SHAPES / "AT_ROI.geojson"             # 4 MGRS tiles, cut into grid cells: the inference case
WORKLOADS = {"fields": "field_id", "cells": "id"}   # workload -> its id column

STARTDATE = datetime.datetime(2018, 4, 1)
ENDDATE = datetime.datetime(2018, 10, 1)
MOSAIC_DAYS = 20                            # 10 calendar windows
WINDOWS = pd.date_range(STARTDATE, freq=f"{MOSAIC_DAYS}D",
                        periods=math.ceil((ENDDATE - STARTDATE).days / MOSAIC_DAYS))
BANDS = ["B04", "B08", "SCL"]               # what rise-eo downloads and builds from
CUBE_BANDS = ["B04", "B08"]                 # what a cube holds once SCL has masked it
COLLECTION = "sentinel-2-l2a"
PROCESSING = "latest"
SCL_CLASSES = list(rise_collections.get(COLLECTION).mask_spec.classes)   # rise-eo's own mask

MAX_TILES = 300          # download guard: Apr-Sep over 4 tiles is about 184 granules
MAX_CONCURRENT = 32      # band files in flight during the download (MPC is in the same region)
CORES = max(1, os.cpu_count() // 2)   # parallel rise-eo builds; each holds a few GB (estimate)
GRID_CRS = "EPSG:32633"  # every Austria tile is in UTM 33N, the zone rise-eo builds in here
RES = 10                 # metres, rise-eo's reference band (B08)
MIN_AGREEMENT = 0.99     # PASS bar for the pixel checks in Step 8: a first guess, not a spec

SHAPES_OUT, TIMINGS, ARCHIVE = OUT / "shapes", OUT / "timings", OUT / "archive"
CATALOG = archive_catalog_filepath(str(ARCHIVE), COLLECTION)


def mpc_config(workload):
    """The mpc-fast-extract config that mirrors rise-eo's build for one workload."""
    return {
        "geometries": str(SHAPES_OUT / f"{workload}.geojson"),
        "id_field": WORKLOADS[workload],
        "start": STARTDATE.date().isoformat(),
        "end": ENDDATE.date().isoformat(),
        "period": f"{MOSAIC_DAYS}d",            # windows anchored at start, like rise-eo's calendar
        "all_touched": True,                    # rise-eo crops with all_touched=True
        "sentinel2": {
            "bands": CUBE_BANDS,
            "composite": "median",              # rise-eo's only mosaic method
            "fuse": "first_valid",              # rise-eo merges one day's granules first-valid
            "cloud": {"scl_mask": SCL_CLASSES,
                      "max_scene_cloud_cover": 100},   # rise-eo drops no scene for cloud cover
        },
        "grid": {"crs": GRID_CRS, "resolution": RES},  # defaults: nearest, pixel edges on 10 m
        "output": {"path": str(OUT / "mpc" / workload / "engine"), "stats": False,
                   "rasters": {"format": "zarr",
                               "dtype": "float32"}},    # float32: DN comes back exactly (x 1e4)
    }

## Helpers

Small functions several steps use. Run this cell after Settings, and again after a restart.

In [ ]:
# Helpers
import json
import time

import geopandas as gpd
import mpc_fast_extract as mfe
import numpy as np


def timed(name, fn):
    """Run fn() unless TIMINGS/<name>.json exists; save its wall seconds (and fn's dict) there."""
    path = TIMINGS / f"{name}.json"
    if path.exists():
        print(name, "already timed:", path.read_text())
        return
    start = time.perf_counter()
    extra = fn() or {}
    seconds = round(time.perf_counter() - start, 1)
    path.write_text(json.dumps({"seconds": seconds, **extra}, default=str))
    print(name, path.read_text())


def read_timing(name):
    """TIMINGS/<name>.json as a dict, or None when that step has not finished."""
    path = TIMINGS / f"{name}.json"
    return json.loads(path.read_text()) if path.exists() else None


def shapes(workload):
    """The workload's geometries as Step 2 wrote them (EPSG:4326)."""
    return gpd.read_file(SHAPES_OUT / f"{workload}.geojson")


def rise_cubes(workload):
    """id -> datacube.npy of every rise-eo cube of the workload that is on disk."""
    rows = pd.read_csv(OUT / "rise" / workload / "input.csv", dtype={"id": str})
    return {i: Path(p) for i, p in zip(rows["id"], rows["datacube_filepath"]) if Path(p).exists()}


def mpc_cubes(workload):
    """id -> datacube.npy of every converted mpc-fast-extract cube of the workload."""
    return {p.parent.name: p for p in (OUT / "mpc" / workload / "cubes").glob("*/datacube.npy")}

## Step 1 — check the machine

**What it does:** checks the machine can run mpc-fast-extract and has room, and records what it is.
**PASS if:** Linux x86-64 with AVX2 and FMA; `mpc_fast_extract` is imported from
`MPC_FAST_EXTRACT` at `MPC_FAST_EXTRACT_COMMIT`; `OUT` is not on the network share; and, before
the first download, `OUT` has at least `NEED_GB` free.
**Time / disk:** seconds.

In [ ]:
# Step 1 -- check the machine, and record it.
import platform
import re
import shutil
import subprocess


def git_head(folder):
    """The short commit a checkout is at."""
    return subprocess.run(["git", "rev-parse", "--short=7", "HEAD"], cwd=folder,
                          capture_output=True, text=True, check=True).stdout.strip()


assert platform.system() == "Linux" and platform.machine() == "x86_64", (
    f"this is {platform.system()} {platform.machine()}; mpc-fast-extract needs Linux x86-64")
cpuinfo = Path("/proc/cpuinfo").read_text()
assert re.search(r"\bavx2\b", cpuinfo) and re.search(r"\bfma\b", cpuinfo), (
    "this CPU lacks AVX2 or FMA, which the mpc-fast-extract build targets")
assert Path(mfe.__file__).resolve().is_relative_to(MPC_FAST_EXTRACT.resolve()), (
    f"mpc_fast_extract is imported from {mfe.__file__}, not from {MPC_FAST_EXTRACT}")
assert git_head(MPC_FAST_EXTRACT) == MPC_FAST_EXTRACT_COMMIT, (
    f"mpc-fast-extract is at {git_head(MPC_FAST_EXTRACT)}, not {MPC_FAST_EXTRACT_COMMIT}")
TIMINGS.mkdir(parents=True, exist_ok=True)
assert "cloudfiles" not in str(OUT.resolve()), f"{OUT} is on the network share; pick a local disk"
free_gb = shutil.disk_usage(OUT).free / 1e9
if not ARCHIVE.exists():
    assert free_gb >= NEED_GB, f"only {free_gb:.0f} GB free at {OUT}; want {NEED_GB}"

machine = {
    "cpu_model": re.search(r"model name\s*:\s*(.+)", cpuinfo).group(1),
    "cpus": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "rise_eo": git_head(CHECKOUT),
    "mpc_fast_extract": git_head(MPC_FAST_EXTRACT),
}
(TIMINGS / "machine.json").write_text(json.dumps(machine))
print(machine, f"| {free_gb:.0f} GB free at {OUT}")

## Step 2 — write the two workloads' geometries

**What it does:** writes `OUT/shapes/fields.geojson` (the 900 fields, id `field_id`) and
`OUT/shapes/cells.geojson` (`AT_ROI` cut into grid cells by `rise_eo.grid.roi_to_s2_grids`, id
`id`), both in EPSG:4326. Both tools read these same files.
**PASS if:** 900 fields; at least one cell; ids unique in each file; everything lies in UTM 33N
(longitude 12° to 18° E).
**Time / disk:** seconds.

In [ ]:
# Step 2 -- write both workloads' geometries.
from rise_eo.grid import roi_to_s2_grids

SHAPES_OUT.mkdir(parents=True, exist_ok=True)
fields = gpd.read_file(LABELS).to_crs(4326)
fields = fields.assign(field_id=fields["fid"].astype(str))[["field_id", "geometry"]]
cells = roi_to_s2_grids(gpd.read_file(ROI))[["id", "geometry"]]
fields.to_file(SHAPES_OUT / "fields.geojson", driver="GeoJSON")
cells.to_file(SHAPES_OUT / "cells.geojson", driver="GeoJSON")

In [ ]:
# Step 2 -- check it.
for workload, id_col in WORKLOADS.items():
    gdf = shapes(workload)
    assert len(gdf) > 0 and gdf[id_col].is_unique, f"{workload}: ids must be present and unique"
    west, _, east, _ = gdf.total_bounds
    assert 12 <= west and east <= 18, f"{workload}: spans {west:.2f}..{east:.2f} E, outside UTM 33N"
    print(workload, len(gdf), "geometries")
assert len(shapes("fields")) == 900, f"expected 900 fields, found {len(shapes('fields'))}"

## Step 3 — plan both sides (nothing is downloaded)

**What it does:** asks MPC which granules rise-eo would download for each workload, and asks
mpc-fast-extract what it would read (`mpc_fast_extract.plan`: tiles, scenes, blocks, bytes). It
saves both to `OUT/timings/plan.json` and prints mpc-fast-extract's plan tables.
**PASS if:** the granule count is between 1 and `MAX_TILES`, and, before the first download, the
estimated archive size fits in the free space at `OUT`.
**Time / disk:** under a minute.

In [ ]:
# Step 3 -- plan both sides.
from rise_eo.sources import mpc as mpc_source

GB_PER_GRANULE = 0.37   # estimate: the 2018 Austria archive held 184 granules of B04/B08/SCL in 67.2 GB
plan = {}
for workload in WORKLOADS:
    granules = mpc_source.query_catalog(shapes(workload), STARTDATE, ENDDATE,
                                        collection=COLLECTION, processing=PROCESSING)
    engine_plan = mfe.plan(mpc_config(workload))
    print(f"--- {workload}: mpc-fast-extract plan\n{engine_plan}")
    plan[workload] = {"granules": sorted(granules["id"]),
                      "engine": engine_plan.summary.reset_index().to_dict("records")}
(TIMINGS / "plan.json").write_text(json.dumps(plan, default=str))

In [ ]:
# Step 3 -- check it.
plan = read_timing("plan")
all_granules = set().union(*(plan[w]["granules"] for w in WORKLOADS))
estimate_gb = len(all_granules) * GB_PER_GRANULE
for workload in WORKLOADS:
    print(workload, len(plan[workload]["granules"]), "granules")
print(len(all_granules), f"granules in all, about {estimate_gb:.0f} GB (estimate)")
assert 0 < len(all_granules) <= MAX_TILES, f"{len(all_granules)} granules; MAX_TILES is {MAX_TILES}"
if not ARCHIVE.exists():
    free_gb = shutil.disk_usage(OUT).free / 1e9
    assert estimate_gb < free_gb, f"about {estimate_gb:.0f} GB to download, {free_gb:.0f} GB free"

## Step 4 — (a) rise-eo downloads the granules

**What it does:** `rise_eo.download` copies every whole granule both workloads need (B04, B08,
SCL) into one archive at `OUT/archive`, the fields' first, then the cells' (which skips what is
already there). The whole download is timed once; the Summary splits it between the workloads by
granule count.
**PASS if:** the catalog exists and the timing file was written.
**Time / disk:** an estimate of 10–30 min; about 70 GB.

In [ ]:
# Step 4 -- (a) download, timed.
def download_both():
    """Download every granule both workloads need; return the archive's size on disk."""
    for workload in WORKLOADS:
        rise_eo.download(
            roi=shapes(workload), startdate=STARTDATE, enddate=ENDDATE, bands=BANDS,
            dst_folderpath=str(ARCHIVE), source="mpc", collection=COLLECTION,
            processing=PROCESSING, max_tiles=MAX_TILES, max_concurrent=MAX_CONCURRENT)
    size = sum(p.stat().st_size for p in ARCHIVE.rglob("*") if p.is_file())
    return {"archive_gb": round(size / 1e9, 2)}


timed("rise_download", download_both)

In [ ]:
# Step 4 -- check it.
assert Path(CATALOG).exists(), f"no catalog at {CATALOG}"
assert read_timing("rise_download") is not None, "the download did not finish; run Step 4 again"
print(read_timing("rise_download"))

## Step 5 — (b) rise-eo builds the cubes

**What it does:** for each workload, `rise_eo.workflows.create_datacube.run_create_datacube`
builds one cube per geometry from the archive with the local runner, `CORES` builds at a time,
from scratch (`overwrite=True`). Each workload is timed separately.
**PASS if:** every geometry listed in the workload's `input.csv` has its cube on disk. (A geometry
with no granules at all is left out of `input.csv` by rise-eo, and printed.)
**Time / disk:** an estimate of 15–40 min for both; a few GB.

In [ ]:
# Step 5 -- (b) build, timed per workload.
from rise_eo.workflows.create_datacube import run_create_datacube


def rise_build(workload):
    """Build every cube of the workload from scratch with the local runner."""
    run = OUT / "rise" / workload
    run.mkdir(parents=True, exist_ok=True)
    run_create_datacube(
        catalog_filepath=CATALOG, timestamp_col="timestamp",
        shapefilepath=str(SHAPES_OUT / f"{workload}.geojson"), id_col=WORKLOADS[workload],
        run_folderpath=str(run), startdate=STARTDATE, enddate=ENDDATE, bands=BANDS,
        mosaic_days=MOSAIC_DAYS, csv_filepath=str(run / "input.csv"), label_col=None,
        cores=CORES, overwrite=True, collection=COLLECTION, processing=PROCESSING)
    return {"cores": CORES, "cubes": len(rise_cubes(workload))}


for workload in WORKLOADS:
    timed(f"rise_build_{workload}", lambda w=workload: rise_build(w))

In [ ]:
# Step 5 -- check it.
for workload in WORKLOADS:
    rows = pd.read_csv(OUT / "rise" / workload / "input.csv", dtype={"id": str})
    missing = sorted(set(rows["id"]) - set(rise_cubes(workload)))
    assert not missing, f"{workload}: {len(missing)} cubes missing, e.g. {missing[:3]}"
    print(workload, len(rows), "cubes;", read_timing(f"rise_build_{workload}"))

## Step 6 — (c1) mpc-fast-extract composites each workload in one run

**What it does:** one `mpc_fast_extract.run` per workload, over all its geometries at once, with
`mpc_config` from Settings. It reads the imagery straight from MPC (no archive) and writes one
Zarr `(time, band, y, x)` of 20-day medians on the UTM 33N 10 m grid. Each run is timed.
**PASS if:** each run's `issues.json` says the output is complete, and its Zarr exists.
**Time / disk:** an estimate of a few minutes per workload; a few GB.

In [ ]:
# Step 6 -- (c1) one engine run per workload, timed.
def mpc_run(workload):
    """One engine run over all the workload's geometries; return the engine's own summary."""
    run = mfe.run(mpc_config(workload), no_resume=True)
    return {"engine": run.summary, "complete": run.complete}


for workload in WORKLOADS:
    timed(f"mpc_run_{workload}", lambda w=workload: mpc_run(w))

In [ ]:
# Step 6 -- check it.
for workload in WORKLOADS:
    run = mfe.Run(mpc_config(workload), mpc_config(workload)["output"]["path"])
    assert run.complete, f"{workload}: issues.json says the output is incomplete: {run.issues}"
    assert Path(run.zarr_path()).exists(), f"{workload}: no Zarr at {run.zarr_path()}"
    print(workload, read_timing(f"mpc_run_{workload}"))

## Step 7 — (c2) cut the Zarr into rise-eo cubes

**What it does:** for each geometry, cuts its bounding box out of the workload's Zarr and writes
it the way rise-eo lays a cube out: `datacube.npy` `(time, height, width, bands)`, uint16 DN
(reflectance × 10,000), nodata 0, pixels outside the geometry set to 0 (touched pixels kept, as
rise-eo does), every window of `WINDOWS` present; plus `metadata.pickle.npy` with the grid, the
window starts and the band names. This is the work a real backend would add on top of the engine,
so it is timed and counted with it.
**PASS if:** every period in the Zarr is one of rise-eo's windows, and every geometry has a cube.
**Time / disk:** an estimate of a minute or two; a few GB.

In [ ]:
# Step 7 -- (c2) convert, timed per workload.
from affine import Affine
from rasterio.crs import CRS
from rasterio.features import geometry_mask

from rise_eo.storage import fs


def window(geom, xs, ys):
    """Row and column slices of the grid pixels whose 10 m cell meets geom's bounding box."""
    minx, miny, maxx, maxy = geom.bounds
    cols = np.flatnonzero((xs + RES / 2 > minx) & (xs - RES / 2 < maxx))
    rows = np.flatnonzero((ys + RES / 2 > miny) & (ys - RES / 2 < maxy))
    return slice(rows[0], rows[-1] + 1), slice(cols[0], cols[-1] + 1)


def save_cube(folder, cube, transform):
    """Write datacube.npy + metadata.pickle.npy the way rise-eo's builder lays them out."""
    folder.mkdir(parents=True, exist_ok=True)
    np.save(folder / "datacube.npy", cube)
    profile = {"crs": CRS.from_string(GRID_CRS), "transform": transform, "nodata": 0,
               "height": cube.shape[1], "width": cube.shape[2], "count": 1, "dtype": "uint16"}
    meta = {"geotiff_metadata": profile, "timestamps": list(WINDOWS), "bands": CUBE_BANDS}
    fs.save_npy(str(folder / "metadata.pickle.npy"), meta, allow_pickle=True)


def mpc_convert(workload):
    """One cube per geometry from the run's Zarr; return the periods the Zarr held."""
    da = mfe.Run(mpc_config(workload), mpc_config(workload)["output"]["path"]).zarr()
    held = [str(t)[:10] for t in da["time"].values]
    da = da.sel(band=CUBE_BANDS).reindex(time=WINDOWS).transpose("time", "y", "x", "band")
    data, xs, ys = da.values, da["x"].values, da["y"].values
    assert ys[0] > ys[-1], "expected a north-up grid"
    gdf = shapes(workload).to_crs(GRID_CRS)
    for cube_id, geom in zip(gdf[WORKLOADS[workload]], gdf.geometry):
        r, c = window(geom, xs, ys)
        transform = Affine(RES, 0, xs[c.start] - RES / 2, 0, -RES, ys[r.start] + RES / 2)
        cube = np.nan_to_num(data[:, r, c, :] * 1e4, nan=0).round().astype("uint16")
        cube[:, geometry_mask([geom], cube.shape[1:3], transform, all_touched=True), :] = 0
        save_cube(OUT / "mpc" / workload / "cubes" / str(cube_id), cube, transform)
    return {"zarr_periods": held, "cubes": len(gdf)}


for workload in WORKLOADS:
    timed(f"mpc_convert_{workload}", lambda w=workload: mpc_convert(w))

In [ ]:
# Step 7 -- check it.
expected = [str(w.date()) for w in WINDOWS]
for workload in WORKLOADS:
    held = read_timing(f"mpc_convert_{workload}")["zarr_periods"]
    extra = sorted(set(held) - set(expected))
    assert not extra, f"{workload}: the Zarr holds periods {extra} that are not rise-eo windows {expected}"
    assert len(mpc_cubes(workload)) == len(shapes(workload)), f"{workload}: some cubes are missing"
    print(workload, len(mpc_cubes(workload)), "cubes; windows without data:",
          sorted(set(expected) - set(held)))

## Step 8 — compare the two cubes of every geometry

**What it does:** for each geometry with both cubes, lines them up by their transforms (they must
share one 10 m pixel lattice) and compares the pixels both cover, over every window and band. It
counts: same shape and grid, same window dates, pixels where exactly one side is nodata, and,
where both have a value, pixels that are equal and pixels within 1 DN (a median of an even count
can round either way). Saved to `OUT/compare/<workload>.json`.
**PASS if,** for each workload:
- no geometry has a rise-eo cube without an mpc-fast-extract cube;
- every compared pair has the same window dates;
- nodata agrees on at least `MIN_AGREEMENT` of the pixels;
- values agree within 1 DN on at least `MIN_AGREEMENT` of the pixels where both have one.

`MIN_AGREEMENT` is a first guess. A miss is a finding to look at, not a broken run.
**Time / disk:** a few minutes.

In [ ]:
# Step 8 -- the comparison helpers.
def load_cube(path):
    """(cube with CUBE_BANDS in that order, its transform, its CRS, its window dates)."""
    meta = fs.load_npy(str(path.parent / "metadata.pickle.npy"), allow_pickle=True)[()]
    cube = np.load(path)[..., [list(meta["bands"]).index(b) for b in CUBE_BANDS]]
    dates = [str(pd.Timestamp(t).date()) for t in meta["timestamps"]]
    profile = meta["geotiff_metadata"]
    return cube, profile["transform"], CRS.from_user_input(profile["crs"]), dates


def overlap(a, ta, b, tb):
    """The parts of cubes a and b (t, H, W, bands) over the same pixels, found by their transforms."""
    dc, dr = (tb.c - ta.c) / RES, (ta.f - tb.f) / RES
    assert abs(dc - round(dc)) < 1e-6 and abs(dr - round(dr)) < 1e-6, "not one pixel lattice"
    dc, dr = round(dc), round(dr)
    r0, c0 = max(0, dr), max(0, dc)
    r1, c1 = min(a.shape[1], dr + b.shape[1]), min(a.shape[2], dc + b.shape[2])
    return a[:, r0:r1, c0:c1], b[:, r0 - dr:r1 - dr, c0 - dc:c1 - dc]


def compare(workload):
    """Agreement counts over every geometry with both cubes, saved to OUT/compare/<workload>.json."""
    rise, mpc = rise_cubes(workload), mpc_cubes(workload)
    n = dict.fromkeys(["same_grid", "same_windows", "pixels", "one_nodata", "both_valid",
                       "equal", "within_1"], 0)
    n.update(compared=len(set(rise) & set(mpc)), only_rise=len(set(rise) - set(mpc)),
             only_mpc=len(set(mpc) - set(rise)))
    for cube_id in sorted(set(rise) & set(mpc)):
        a, ta, crs, wa = load_cube(rise[cube_id])
        b, tb, _, wb = load_cube(mpc[cube_id])
        assert crs == CRS.from_string(GRID_CRS), f"{cube_id}: rise-eo built it in {crs}"
        n["same_grid"] += a.shape == b.shape and ta == tb
        n["same_windows"] += wa == wb
        a, b = overlap(a, ta, b, tb)
        both, diff = (a > 0) & (b > 0), np.abs(a.astype(np.int32) - b)
        n["pixels"] += a.size
        n["one_nodata"] += int(((a > 0) != (b > 0)).sum())
        n["both_valid"] += int(both.sum())
        n["equal"] += int((diff[both] == 0).sum())
        n["within_1"] += int((diff[both] <= 1).sum())
    (OUT / "compare").mkdir(exist_ok=True)
    (OUT / "compare" / f"{workload}.json").write_text(json.dumps(n))
    return n


for workload in WORKLOADS:
    print(workload, compare(workload))

In [ ]:
# Step 8 -- check it.
for workload in WORKLOADS:
    n = json.loads((OUT / "compare" / f"{workload}.json").read_text())
    nodata_agree = 1 - n["one_nodata"] / n["pixels"]
    within_1 = n["within_1"] / n["both_valid"]
    print(f"{workload}: nodata agrees on {nodata_agree:.4%}, values within 1 DN on {within_1:.4%}"
          f", equal on {n['equal'] / n['both_valid']:.4%}")
    assert n["only_rise"] == 0, f"{workload}: {n['only_rise']} rise-eo cubes have no counterpart"
    assert n["same_windows"] == n["compared"], f"{workload}: window dates differ on some cubes"
    assert nodata_agree >= MIN_AGREEMENT, f"{workload}: nodata agrees on only {nodata_agree:.4%}"
    assert within_1 >= MIN_AGREEMENT, f"{workload}: values within 1 DN on only {within_1:.4%}"

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart (run Setup, Settings and Helpers first),
and after a failed check. A step that has not finished shows `None`.

- `rise_download_s_est`: the download time split by this workload's share of the granules (an
  estimate: the two workloads were downloaded into one archive).
- `ratio_build_only`: rise-eo's build time ÷ mpc-fast-extract's (run + conversion), as if the
  archive were already on disk.
- `ratio_with_download`: the same, counting rise-eo's estimated download time too.

In [ ]:
import json


def seconds(name):
    """A timed step's wall seconds, or None when it has not finished."""
    t = read_timing(name)
    return t and t["seconds"]


def ratio(a, b):
    """a / b rounded, or None when either is missing."""
    return round(a / b, 1) if a and b else None


plan = read_timing("plan") or {}
all_granules = set().union(*(plan.get(w, {}).get("granules", []) for w in WORKLOADS))
summary = {"machine": read_timing("machine"), "cores_for_rise_builds": CORES,
           "rise_download": read_timing("rise_download")}
for workload in WORKLOADS:
    granules = plan.get(workload, {}).get("granules", [])
    share = len(granules) / len(all_granules) if all_granules else None
    download = seconds("rise_download") and share and round(seconds("rise_download") * share, 1)
    build = seconds(f"rise_build_{workload}")
    run, convert = seconds(f"mpc_run_{workload}"), seconds(f"mpc_convert_{workload}")
    mpc_total = run and convert and run + convert
    compared = OUT / "compare" / f"{workload}.json"
    summary[workload] = {
        "granules": len(granules),
        "rise_download_s_est": download, "rise_build_s": build,
        "mpc_run_s": run, "mpc_convert_s": convert,
        "ratio_build_only": ratio(build, mpc_total),
        "ratio_with_download": ratio(download and build and download + build, mpc_total),
        "mpc_engine": (read_timing(f"mpc_run_{workload}") or {}).get("engine"),
        "compare": json.loads(compared.read_text()) if compared.exists() else None,
    }
print(json.dumps(summary, indent=2, default=str))